In [11]:
#!/usr/bin/env python3
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
import sys
sys.path.append('../')
from helper import varDict,getTitleFromDF

try:
    import mplhep as hep
    hep.style.use("ATLAS")
except:
    pass


## Get Jack's numbers

In [12]:
jackDF = pd.read_csv('../../jacksFiles/B2F_page2_table.csv',index_col=0)
jackDF = jackDF.rename(columns={'mS_GeV':'ms', 'm1_GeV':'m1', 'Delta_m10':'dm','sigma_fb' : 'xsec_fb'})
jackDF.drop(columns=['beta_chi1_scalar_rest_frame','N_Run2_plus_3'], inplace=True)

In [13]:
jackDF

,ms,m1,dm,xsec_fb,N_Run2
id,,,,,
100004,450,215.5,60,32.78,1.57
100008,450,204.1,60,74.47,1.53
100006,450,212.4,60,46.63,1.29
100016,600,283.9,60,8.33,1.15
100009,450,204.1,100,71.43,0.98
100007,450,212.4,100,44.91,0.92
100018,600,272.9,60,14.49,0.90
100014,600,288.1,60,5.81,0.76
100005,450,215.5,100,31.51,0.65


### Get B2TF recast data

In [15]:
b2tf_effs_R = np.genfromtxt('../b2tf_new_effs.csv',delimiter=',',names=True)
b2tf_data_df_R = pd.DataFrame(b2tf_effs_R,columns=b2tf_effs_R.dtype.names)
b2tf_data_df_R['dm'] = b2tf_data_df_R['m1']-b2tf_data_df_R['m0']
b2tf_data_df_R['ctau'] = (b2tf_data_df_R['tau']*2.99e8).round(0)
b2tf_data_df_R['ctau_full'] = (b2tf_data_df_R['tau']*2.99e8)
b2tf_data_df_R['xsec_fb'] = (b2tf_data_df_R['xsec_pb']*1e3)
b2tf_data_df_R['N_Run2'] = b2tf_data_df_R['eff']*b2tf_data_df_R['xsec_fb']*139.0
b2tf_data_df_R['N_Run2_err'] = b2tf_data_df_R['efferror']*b2tf_data_df_R['xsec_fb']*139.0

In [16]:
b2tf_data_df_R['ms'].unique(),b2tf_data_df_R['ctau'].unique()

(array([ 400.,  450.,  600.,  800., 1000.]), array([20.]))

### Add ratio to Jack's Numbers

In [30]:
# Round the keys before matching: dm = m1 - m0 has floating-point noise,
# and Jack's masses are given to 1 decimal place
keys = ['ms', 'm1', 'dm']
jack_keyed = jackDF[keys + ['xsec_fb', 'N_Run2']].copy()
jack_keyed[keys] = jack_keyed[keys].round(1)

b2tf_keyed = b2tf_data_df_R.copy()
b2tf_keyed[keys] = b2tf_keyed[keys].round(1)

merged = b2tf_keyed.merge(jack_keyed, on=keys, how='left',
                          suffixes=('', '_jack'), validate='many_to_one')

b2tf_data_df_R['N_Run2_jack'] = merged['N_Run2_jack'].values
b2tf_data_df_R['xsec_jack'] = merged['xsec_fb_jack'].values
b2tf_data_df_R['N_Run2_ratio'] = b2tf_data_df_R['N_Run2'] / b2tf_data_df_R['N_Run2_jack']
b2tf_data_df_R['N_Run2_ratio_err'] = b2tf_data_df_R['N_Run2_err'] / b2tf_data_df_R['N_Run2_jack']



In [32]:
b2tf_match = b2tf_data_df_R[b2tf_data_df_R['N_Run2_ratio'].notnull()]
b2tf_match = b2tf_match.sort_values(by=['N_Run2_jack'], ascending=False)
# b2tf_match[['ms', 'm1', 'dm', 'N_Run2', 'N_Run2_err', 'N_Run2_jack', 'N_Run2_ratio', 'N_Run2_ratio_err']]
b2tf_match[['ms', 'm1', 'dm', 'N_Run2', 'N_Run2_jack', 'N_Run2_ratio']]

,ms,m1,dm,N_Run2,N_Run2_jack,N_Run2_ratio
14,450.0,215.5,60.0,1.653570,1.57,1.053229
18,450.0,204.1,60.0,1.402336,1.53,0.916559
16,450.0,212.4,60.0,2.933758,1.29,2.274231
26,600.0,283.9,60.0,1.966221,1.15,1.709757
19,450.0,204.1,100.0,4.932932,0.98,5.033604
17,450.0,212.4,100.0,0.561762,0.92,0.610610
28,600.0,272.9,60.0,2.134662,0.90,2.371847
24,600.0,288.1,60.0,1.222510,0.76,1.608565
15,450.0,215.5,100.0,0.395216,0.65,0.608024
25,600.0,288.1,100.0,1.141499,0.61,1.871310


In [33]:
b2tf_match[['ms', 'm1', 'dm', 'xsec_fb', 'xsec_jack']]

,ms,m1,dm,xsec_fb,xsec_jack
14,450.0,215.5,60.0,18.690,32.78
18,450.0,204.1,60.0,42.230,74.47
16,450.0,212.4,60.0,26.880,46.63
26,600.0,283.9,60.0,5.191,8.33
19,450.0,204.1,100.0,40.540,71.43
17,450.0,212.4,100.0,25.660,44.91
28,600.0,272.9,60.0,9.055,14.49
24,600.0,288.1,60.0,3.606,5.81
15,450.0,215.5,100.0,17.950,31.51
25,600.0,288.1,100.0,3.468,5.58
